In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import gridspec
from data_setup import RotatedMNIST, create_dataloaders
from model import MultiTaskDNN

In [ ]:
# 1. Create a mock config to test the pipeline locally
mock_data_cfg = {
    'dataset_path': './DNN_Data',
    'batch_size': 32,
    'num_workers': 0,  # Set to 0 to avoid multiprocessing issues during plotting
    'max_digit': 9,
    'max_rotation_degrees': 360
}

print("Running Data Setup Sanity Check...")

# 2. Get the dataloaders
train_loader, _ = create_dataloaders(mock_data_cfg)

# 3. Grab a single batch from the training loader
images, digits, angle_rad, angle_deg = next(iter(train_loader))

# 4. Set up the matplotlib figure
fig, axes = plt.subplots(1, 5, figsize=(15, 4))

for i in range(5):
    # Extract the image and remove the channel dimension [1, 28, 28] -> [28, 28]
    img = images[i].squeeze().numpy()

    # Extract values for this particular image
    digit = digits[i].item()
    angle = angle_deg[i].item()

    # Plot the image and set the title
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Digit: {digit}\nAngle: {angle:.0f}°")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print("Running Model Architecture Sanity Check...")

mock_model_cfg = {
    'input_shape': 784,
    'shared_layers': 2,
    'task_layers': 2,
    'shared_nodes': 16,
    'task_nodes': 16,
    'digit_out': 10,
    'parity_out': 2,
    'rotation_out': 1
}

mock_train_cfg = {
    'activation': 'relu'
}

# Initialize model
model = MultiTaskDNN(mock_model_cfg, mock_train_cfg)

# Create a dummy batch of 5 images (28x28)
dummy_input = torch.randn(5, 1, 28, 28)

# Pass through the model
digit_preds, oddeven_preds, rot_preds, acts = model(dummy_input)

print(f"Model successfully processed dummy batch of shape {dummy_input.shape}")
print(f"Digit Output Shape: {digit_preds.shape} (Expected: 5, 10)")
print(f"Odd/Even Output Shape: {oddeven_preds.shape} (Expected: 5, 2)")
print(f"Rotation Output Shape: {rot_preds.shape} (Expected: 5, 1)")

print("\nTracked Activations:")
for key, tensor in acts.items():
    print(f"  {key}: shape {tensor.shape}")

## Training

In [ ]:
# Training configs
training_data_cfg = {
    'dataset_path': './DNN_Data',
    'batch_size': 32,
    'num_workers': 0,  # Set to 0 to avoid multiprocessing issues during plotting
    'max_digit': 9,
    'max_rotation_degrees': 360
}

training_model_cfg = {
    'input_shape': 784,
    'shared_layers': 5,
    'task_layers': 5,
    'shared_nodes': 16,
    'task_nodes': 16,
    'digit_out': 10,
    'parity_out': 2,
    'rotation_out': 1
}

train_cfg = {
    'activation': 'relu'
}

In [ ]:
# Full training routine
train_loader, test_loader = create_dataloaders(training_data_cfg)
model = MultiTaskDNN(training_model_cfg, train_cfg)

# Loss functions for each head
digit_loss = nn.CrossEntropyLoss()
parity_loss = nn.CrossEntropyLoss()
def angle_loss(pred, target):
    # Match [B] shapes and measure the shortest signed angular difference.
    diff = pred.squeeze(-1) - target
    circular_diff = torch.atan2(torch.sin(diff), torch.cos(diff))
    return circular_diff.square().mean()

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
model.train()
epochs = 25
epoch_losses = []
print("Start of training!")
for epoch in range(epochs):
    # Reset epoch loss
    epoch_loss = 0.0
    epoch_digit_loss, epoch_parity_loss, epoch_rot_loss = 0.0, 0.0, 0.0
    
    for images, digits, angle_rad, angle_deg in train_loader:
        # Reset gradients
        optimizer.zero_grad()
        # Determine parity labels
        parity = digits % 2

        # Make predictions
        digit_preds, parity_preds, rot_preds, acts = model(images)

        # Assess loss on each head, then sum together
        l_digit = digit_loss(digit_preds, digits)
        l_parity = parity_loss(parity_preds, parity)
        l_rot = angle_loss(rot_preds, angle_rad)

        full_loss = l_digit + l_parity + l_rot
        
        # Make gradient step backwards
        full_loss.backward()
        optimizer.step()

        # Record losses
        epoch_loss += full_loss.item()
        epoch_digit_loss += l_digit.item()
        epoch_parity_loss += l_parity.item()
        epoch_rot_loss += l_rot.item()

    # At end of each epoch, report average batch losses
    avg_loss = epoch_loss / len(train_loader)
    avg_digit_loss = epoch_digit_loss / len(train_loader)
    avg_parity_loss = epoch_parity_loss / len(train_loader)
    avg_rot_loss = epoch_rot_loss / len(train_loader)
    print(f"   epoch {epoch+1}: avg batch loss {avg_loss:.4f}"
          f"   (digit {avg_digit_loss:.4f}, parity {avg_parity_loss:.4f}, rot {avg_rot_loss:.4f})")
    epoch_losses.append(avg_loss)

## Graphing

In [ ]:
plt.plot(range(epochs), epoch_losses, '-o')
plt.title("Epoch Losses")
plt.xlabel("Epoch")
plt.ylabel("Avg Epoch Loss (Digit + Parity + Rotation)")
plt.show()

In [ ]:
# Evaluate model on held-out test data
total_digit_loss, total_parity_loss, total_rot_loss = 0.0, 0.0, 0.0
total_batches = 0
model.eval()
with torch.no_grad():
    for images, digits, angle_rad, angle_deg in test_loader:
        # Determine parity labels
        parity = digits % 2

        # Make predictions
        digit_preds, parity_preds, rot_preds, acts = model(images)

        # Accumulate losses
        total_digit_loss += digit_loss(digit_preds, digits).item()
        total_parity_loss += parity_loss(parity_preds, parity).item()
        total_rot_loss += angle_loss(rot_preds, angle_rad).item()
    
        total_batches += 1

avg_total_loss = (total_digit_loss + total_parity_loss + total_rot_loss) / total_batches
avg_digit_loss = total_digit_loss / total_batches
avg_parity_loss = total_parity_loss / total_batches
avg_rot_loss = total_rot_loss / total_batches
print(f"avg test batch loss {avg_total_loss:.4f}"
      f"   (digit {avg_digit_loss:.4f}, parity {avg_parity_loss:.4f}, rot {avg_rot_loss:.4f})")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model.eval()
rot_preds, rot_labels = [], []
digit_preds_all, digit_labels = [], []

with torch.no_grad():
    for images, digits, angle_rad, angle_deg in test_loader:
        digit_pred, _, rot_pred, _ = model(images)
        digit_preds_all.append(digit_pred.argmax(dim=1).cpu())
        digit_labels.append(digits.cpu())
        rot_preds.append(rot_pred.cpu())
        rot_labels.append(angle_rad.cpu())
rot_preds_arr = torch.cat(rot_preds).numpy().flatten()
rot_labels_arr = torch.cat(rot_labels).numpy()
cm = confusion_matrix(torch.cat(digit_labels).numpy(),
                      torch.cat(digit_preds_all).numpy(), labels=list(range(10)))


In [ ]:
plt.hist(rot_preds_arr)
plt.show()

In [ ]:
# Rotation plot
rot_preds_arr  = torch.cat(rot_preds).numpy().flatten()
rot_labels_arr = torch.cat(rot_labels).numpy()

diff = rot_preds_arr - rot_labels_arr
circular_err_rad = np.abs(np.arctan2(np.sin(diff), np.cos(diff)))
rot_mse_losses = circular_err_rad ** 2

plt.hist(rot_mse_losses, bins=100)
# Squared circular error is in rad^2; square each radian threshold
for deg, color in [(20, "green"), (45, "yellow"), (90, "orange"), (180, "red")]:
    thresh_rad2 = np.deg2rad(deg) ** 2
    thresh_mean = (rot_mse_losses < thresh_rad2).mean()
    plt.axvline(thresh_rad2, color=color, linestyle="--", label=f"{deg}° diff. ({thresh_mean:.03f})")

plt.legend()
plt.show()

In [ ]:
# Digit Plot
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(cmap="Blues", values_format="d")
plt.title("Digit Confusion Matrix")
plt.show()

In [ ]:
overall_acc = cm.trace() / cm.sum()
print(f"Overall accuracy: {overall_acc:.4f}")

## Partial Derivative Maps

In [ ]:
model.eval()
images, _, _, _ = next(iter(train_loader))
_, _, _, acts = model(images[0:1])
acts.keys()

In [ ]:
def jacobian_acts_wrt_input(model, image, layer_name):
    model.eval()  # Use stored batch-normalization statistics for one image.
    if image.ndim == 3:
        image = image.unsqueeze(0)
    if image.ndim != 4 or image.shape[0] != 1:
        raise ValueError("Expected one image with shape [1, H, W] or [1, 1, H, W]")
    image_copy = image.clone().detach().requires_grad_(True)

    def f(x):
        # x: (1, 1, 28, 28) -> (A,)  — recompute inside, from x
        _, _, _, acts = model(x)
        return acts[layer_name].squeeze(0) # (A,)

    jac = torch.autograd.functional.jacobian(f, image_copy)  # (A, 1, 1, 28, 28)
    return jac[:, 0, 0]                         # (A, 28, 28)

In [ ]:
def plot_jacobians_across_images(layer_name, num_samples):
    images, _, _, _ = next(iter(train_loader))
    num_samples = min(num_samples, len(images))
    jacobians = []
    
    for i in range(num_samples): # (1, 1, 28, 28)
        image = images[i:i + 1]
        jac = jacobian_acts_wrt_input(model, image, layer_name)
        print(jac.shape)
        jacobians.append(jac)
    
    # stack -> (B, A, 28, 28)
    stacked_jacobians = torch.stack(jacobians)
    neuron_count = stacked_jacobians.shape[1]
    vmax = stacked_jacobians.abs().max().item()
    
    for i in range(num_samples):
        ncols_act = 8
        nrows_act = int(np.ceil(neuron_count / ncols_act))
        fig = plt.figure(figsize=(2.2 + ncols_act * 1.5, nrows_act * 1.5))
    
        outer = gridspec.GridSpec(
            1, 2, figure=fig,
            width_ratios=[1, ncols_act],   # input is 1 "cell" wide, acts are ncols_act
            wspace=0.05,
        )
    
        # --- left: original image ---
        ax_in = fig.add_subplot(outer[0])
        ax_in.imshow(images[i, 0].cpu(), cmap="gray")
        ax_in.set_title("input"); ax_in.axis("off")
    
        # --- right: nested grid of activations ---
        inner = gridspec.GridSpecFromSubplotSpec(
            nrows_act, ncols_act, subplot_spec=outer[1],
            wspace=0.05, hspace=0.05,
        )
        im = None
        for j in range(neuron_count):
            r, c = divmod(j, ncols_act)
            ax = fig.add_subplot(inner[r, c])
            im = ax.imshow(stacked_jacobians[i, j], cmap="RdBu_r",
                           vmin=-vmax, vmax=vmax)
            ax.set_title(f"a{j}", fontsize=6, pad=1)
            ax.axis("off")
    
        fig.colorbar(im, ax=fig.axes, fraction=0.02, pad=0.01,
                     label="∂a/∂x")
        fig.suptitle(f"{layer_name} - sample {i}", y=1.02)
        plt.show()

In [ ]:
def plot_jacobians_across_layers(layer_names, image_idx=0):
    images, _, _, _ = next(iter(train_loader))
    image = images[image_idx:image_idx + 1]        # (1, 1, 28, 28)

    # compute per layer
    for layer_name in layer_names:
        jac = jacobian_acts_wrt_input(model, image, layer_name)   # (A, 28, 28)
        stacked = jac.cpu().numpy()                               # (A, 28, 28)

        # one figure per layer 
        neuron_count = stacked.shape[0]
        ncols_act    = 8
        nrows_act    = int(np.ceil(neuron_count / ncols_act))
        vmax         = np.abs(stacked).max()

        fig = plt.figure(figsize=(2.2 + ncols_act * 1.5, nrows_act * 1.5))

        outer = gridspec.GridSpec(
            1, 2, figure=fig,
            width_ratios=[1, ncols_act], wspace=0.05,
        )

        # left: original image (same every figure)
        ax_in = fig.add_subplot(outer[0])
        ax_in.imshow(image[0, 0].cpu(), cmap="gray")
        ax_in.set_title("input"); ax_in.axis("off")

        # right: nested grid
        inner = gridspec.GridSpecFromSubplotSpec(
            nrows_act, ncols_act, subplot_spec=outer[1],
            wspace=0.05, hspace=0.05,
        )
        mosaic_axes = []
        im = None
        for j in range(neuron_count):
            r, c = divmod(j, ncols_act)
            ax = fig.add_subplot(inner[r, c])
            im = ax.imshow(stacked[j], cmap="RdBu_r",
                           vmin=-vmax, vmax=vmax)
            ax.set_title(f"a{j}", fontsize=6, pad=1)
            ax.axis("off")
            mosaic_axes.append(ax)

        fig.colorbar(im, ax=mosaic_axes, fraction=0.02, pad=0.01,
                     label="∂a/∂x")
        fig.suptitle(f"{layer_name} — sample {image_idx}", y=1.02)
        plt.show()

In [ ]:
import time

layer_names = ["Rotation_Layer_0", "Rotation_Layer_4"]

for i in range(5):
    plot_jacobians_across_layers(layer_names, image_idx=i)

In [ ]:
layer_name = "Rotation_Layer_4"
num_samples = 5

plot_jacobians_across_images(layer_name, num_samples)

In [ ]:
def plot_jacobian_tunings(layer_name, num_batches):
    jacobians, sampled_digits = [], []
    if num_batches < 1:
        raise ValueError("num_batches must be positive")

    for batch_idx, (images, digits, _, _) in enumerate(train_loader):
        for i in range(len(images)):
            jac = jacobian_acts_wrt_input(model, images[i:i + 1], layer_name)
            jacobians.append(jac)
        sampled_digits.append(digits)
        if batch_idx + 1 >= num_batches:
            break

    # stack -> (B, A, 28, 28)
    stacked_jacobians = torch.stack(jacobians)
    sampled_digits = torch.cat(sampled_digits)
    neuron_count = stacked_jacobians.shape[1]
    all_digit_tunings = []
    for neuron_idx in range(neuron_count):
        digit_tunings = []
        for digit in range(10):
            mask = sampled_digits == digit
            # Missing digits have no estimated tuning, rather than zero sensitivity.
            tuning = (stacked_jacobians[mask, neuron_idx].abs().mean().item()
                      if mask.any() else float('nan'))
            digit_tunings.append(tuning)
        print(digit_tunings)
        all_digit_tunings.append(digit_tunings)
    return all_digit_tunings

plot_jacobian_tunings("Rotation_Layer_0", 1)
